# StudyLens: train the arrow detector (inline, with progress bars)

Same model and data as `scripts/train_arrow_detector.py`, but the training loop runs **in this notebook** with live progress bars, a loss plot and example predictions. Checkpoints are compatible with the script version: a run started there resumes here (and vice versa).

* **Mask R-CNN** (torchvision, COCO start) learns two classes from AI2D: **arrow** (outline) and **arrowhead** (box). `studylens/arrows.py` then attaches each arrow's tail and head to the nearest part.
* **GPU:** a **T4** is enough. **Time:** roughly 1–2 h for 12 epochs.
* **Speed tip:** AI2D is copied to Colab's local disk first; reading thousands of small files from Google Drive is very slow. Checkpoints stay on Drive, so a disconnect loses at most one epoch.
* **Licences:** AI2D (AI2 non-commercial, no redistribution); torchvision and its COCO weights (BSD). Keep the trained weights private.
* Only the AI2D **train** split is used; 150 train diagrams are held out for validation. The test split is never used.

## 1. Drive and GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/studylens'
!mkdir -p $DRIVE/external $DRIVE/checkpoints
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Code and packages

In [ ]:
REPO_URL = 'https://github.com/krish-patel1003/CECS558-term-project-studylens.git'
import os
if not os.path.exists('/content/studylens'):
    !git clone -q $REPO_URL /content/studylens
%cd /content/studylens
!git pull -q
!pip install -q -r requirements.txt torchvision

## 3. AI2D on the local disk (fast), checkpoints on Drive
If AI2D is already on Drive (from an earlier notebook), it is copied to the local disk (a few minutes, with a progress line). Otherwise it is downloaded (you accept its licence with `--accept-license`) and also saved to Drive for next time.

In [ ]:
!rm -rf external checkpoints   # removes old symlinks only, not the Drive folders
!ln -s $DRIVE/checkpoints checkpoints
!mkdir -p external
if os.path.exists(f'{DRIVE}/external/ai2d/ai2d/categories.json'):
    !rsync -a --info=progress2 $DRIVE/external/ai2d/ external/ai2d/
else:
    !python scripts/get_ai2d.py --accept-license
    !rsync -a external/ai2d/ $DRIVE/external/ai2d/
!ls external/ai2d/ai2d | head && ls external/ai2d/ai2d/images | wc -l

## 4. Settings and data split

In [ ]:
import sys, time, math, json, random
sys.path[:0] = ['/content/studylens', '/content/studylens/scripts']
import torch
import train_arrow_detector as tad
from studylens.arrows import build_model

RUN = 'arrows'      # checkpoints/arrows/ on Drive (same as the script version: resumes it)
EPOCHS = 12
BATCH = 4           # use 2 (and LR = 0.005) if you run out of GPU memory
LR = 0.01
WORKERS = 2
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

t0 = time.time()
train_ids, held_ids = tad.split(150)
print(f'{len(train_ids)} training diagrams, {len(held_ids)} held out  ({time.time() - t0:.0f}s)  device={DEV}')

Check the labels: one training diagram with its arrow outlines (red) and arrowheads (blue).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mp
from PIL import Image
iid = train_ids[3]
img = Image.open(tad.ai2d.AI2D_DIR / 'images' / iid).convert('RGB')
boxes, labels, _ = tad.targets(tad.annotation(iid), img.size)
ann = tad.annotation(iid)
fig, ax = plt.subplots(figsize=(7, 7)); ax.imshow(img); ax.axis('off')
for a in ann['arrows'].values():
    ax.add_patch(mp.Polygon(a['polygon'], fill=False, ec='red', lw=1.5))
for b, l in zip(boxes, labels):
    if l == 2: ax.add_patch(mp.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec='blue', lw=1.5))
ax.set_title(f'{iid}: {labels.count(1)} arrows, {labels.count(2)} arrowheads'); plt.show()

## 5. Train (resumable, live progress)
One progress bar per epoch (running loss, learning rate, speed). After each epoch the checkpoint is saved to Drive and the loss curve is redrawn. If Colab disconnects: re-run sections 1–4, then this cell; it resumes after the last finished epoch.

In [ ]:
from tqdm.auto import tqdm
from IPython.display import clear_output

dl = torch.utils.data.DataLoader(tad.AI2DArrows(train_ids, True), batch_size=BATCH, shuffle=True,
                                 num_workers=WORKERS, collate_fn=tad.collate, drop_last=True)
model = build_model(pretrained=True).to(DEV)
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.SGD(params, lr=LR, momentum=0.9, weight_decay=1e-4)
steps, warm = EPOCHS * len(dl), min(500, len(dl))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(s, steps) / steps)))
scaler = torch.amp.GradScaler(enabled=DEV == 'cuda')
out = tad.CKPT / RUN; out.mkdir(parents=True, exist_ok=True)
start, history = 0, []
if (out / 'last.pt').exists():
    st = torch.load(out / 'last.pt', map_location='cpu')
    model.load_state_dict(st['model']); opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
    start, history = st['epoch'] + 1, st.get('history', [])
    print(f'resuming after epoch {start}')

model.train()
for ep in range(start, EPOCHS):
    run, t0 = 0.0, time.time()
    bar = tqdm(dl, desc=f'epoch {ep + 1}/{EPOCHS}', leave=True)
    for k, (imgs, tgts) in enumerate(bar):
        imgs = [x.to(DEV) for x in imgs]
        tgts = [{kk: v.to(DEV) for kk, v in t.items()} for t in tgts]
        with torch.autocast(device_type='cuda', enabled=DEV == 'cuda'):
            losses = model(imgs, tgts)
            loss = sum(losses.values())
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        run += float(loss.detach())
        if k % 10 == 0:
            bar.set_postfix(loss=f'{run / (k + 1):.3f}', mask=f"{float(losses['loss_mask']):.3f}", lr=f"{sched.get_last_lr()[0]:.4f}")
    history.append(run / len(dl))
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(), 'epoch': ep, 'history': history}, out / 'last.pt')
    torch.save({'model': model.state_dict()}, out / 'arrow_detector.pt')
    plt.figure(figsize=(5, 3)); plt.plot(range(1, len(history) + 1), history, 'o-'); plt.xlabel('epoch'); plt.ylabel('mean loss'); plt.title('training loss'); plt.show()
    print(f'epoch {ep + 1} saved to Drive: loss {history[-1]:.3f}, {(time.time() - t0) / 60:.1f} min')
print('done')

## 6. Validation on the 150 held-out diagrams
For three confidence thresholds: detection precision/recall at IoU 0.5, and **arrow F1 on gold parts** (detected arrows attached to the true parts with StudyLens's rule), next to the same rule on the true arrow outlines (the ceiling). The best threshold's metrics are saved as `checkpoints/arrows/val_metrics.json`.

In [ ]:
import argparse, shutil
rows = {}
for sc in (0.3, 0.5, 0.7):
    print(f'--- score threshold {sc}')
    r = tad.evaluate(argparse.Namespace(run=RUN, n_holdout=150, n_eval=150, score=sc))
    rows[sc] = r
    shutil.copy(out / 'val_metrics.json', out / f'val_metrics_{sc}.json')
clear_output()
print(f"{'score':>6} {'arrow P/R':>12} {'head P/R':>12} {'arrow F1 (detector)':>20} {'ceiling':>8}")
for sc, r in rows.items():
    d, f = r['detection_iou50'], r['arrow_f1_on_gold_parts']
    print(f"{sc:>6} {d['arrow']['precision']:.2f}/{d['arrow']['recall']:.2f}  {d['arrowhead']['precision']:.2f}/{d['arrowhead']['recall']:.2f}  {f['detector']['f1']:>20.3f} {f['gold_outlines']['f1']:>8.3f}")
best = max(rows, key=lambda s: rows[s]['arrow_f1_on_gold_parts']['detector']['f1'])
shutil.copy(out / f'val_metrics_{best}.json', out / 'val_metrics.json')
print(f'best threshold: {best}  (tell Claude this number and the table above)')

## 7. See it work on held-out diagrams
Red: detected arrows; blue: arrowheads; green: the part-to-part arrows StudyLens would add (on the true parts).

In [ ]:
from studylens.arrows import ArrowDetector, link_arrows
det = ArrowDetector(out / 'arrow_detector.pt', score=best)
for iid in held_ids[:4]:
    img = Image.open(tad.ai2d.AI2D_DIR / 'images' / iid).convert('RGB')
    arrows, heads = det.detect(img)
    nodes, _ = tad.gold_nodes(tad.annotation(iid))
    byid = {n.id: n for n in nodes}
    fig, ax = plt.subplots(figsize=(7, 7)); ax.imshow(img); ax.axis('off')
    for a in arrows:
        xs, ys = zip(*a['points']); ax.scatter(xs, ys, s=1, c='red', alpha=0.3)
    for h in heads:
        ax.add_patch(mp.Rectangle((h[0], h[1]), h[2] - h[0], h[3] - h[1], fill=False, ec='blue', lw=1.5))
    for s, t in link_arrows(arrows, heads, nodes, img.size):
        a, b = byid[s].label_bbox, byid[t].label_bbox
        ax.annotate('', xy=((b[0] + b[2]) / 2, (b[1] + b[3]) / 2), xytext=((a[0] + a[2]) / 2, (a[1] + a[3]) / 2),
                    arrowprops=dict(arrowstyle='->', color='lime', lw=2))
    ax.set_title(f'{iid}: {len(arrows)} arrows, {len(heads)} heads'); plt.show()

## 8. Use the detector
Download `checkpoints/arrows/arrow_detector.pt` from Drive (`MyDrive/studylens/checkpoints/arrows/`, about 170 MB) into the project's `checkpoints/` folder on the laptop. The scene graph then uses it automatically instead of the VLM's arrow list (`ARROW_DETECTOR=0` switches it off). Evaluation on the 100 AI2D development diagrams, on the laptop:

```
python scripts/evaluate_grounding.py --n-parts 60 --n-flow 40 --tag v2d
```